# 136 · DeerFlow Sandboxed SWE Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/136-deerflow-sandboxed-swe-agent/swe_sandbox_workbook.ipynb)

**What this notebook teaches:** How a sandboxed Software Engineering agent fixes bugs in an isolated workspace — and why the sandbox *provider* choice matters for safety.

**Two halves:**
- **Part A (cells 1–4):** Pure Python — run anywhere, no server needed. You'll see the bug, understand the sandbox distinction, and step through the fix logic yourself.
- **Part B (cells 5–7):** Requires a running DeerFlow instance. Cells fail fast with a clear message if the server is not reachable.

| Concept | Why it matters |
|---------|----------------|
| `LocalSandboxProvider` | File read/write in the host process — no shell isolation |
| `AioSandboxProvider` | Bash runs inside a containerised async workspace — host is safe |
| Thread-local workspace | Each DeerFlow request gets its own file namespace — no cross-contamination |
| Fail-fast setup | If the runtime is down, tell the student immediately — never silently hang |

In [ ]:
# Part A · Cell 1 — Install
%pip install -q httpx python-dotenv

## Part A · The Bug

Below is the fixture we'll upload to DeerFlow. It has a single intentional bug: `add()` subtracts instead of adds.

```python
# calculator.py  — fixtures/buggy_repo/
def add(a, b):
    return a - b  # BUG: should be a + b

def multiply(a, b):
    return a * b
```

```python
# test_calculator.py
def test_add():
    assert add(2, 3) == 5   # FAILS — add() returns -1

def test_multiply():
    assert multiply(3, 4) == 12  # passes
```

**Why this fixture?** It's the smallest possible real bug — one character wrong, one test failing. The teaching focus is on the *sandbox*, not the complexity of the code.

In [ ]:
# Part A · Cell 2 — Verify the bug runs locally (no DeerFlow needed)

def buggy_add(a, b):
    return a - b  # the bug

def fixed_add(a, b):
    return a + b  # the fix

print('Before fix:')
print(f'  add(2, 3) = {buggy_add(2, 3)}  ✗  (expected 5)')
print(f'  test_add: FAIL')

print('\nAfter fix:')
print(f'  add(2, 3) = {fixed_add(2, 3)}  ✓')
print(f'  test_add: PASS')

## Part A · Sandbox Provider Comparison

DeerFlow supports two sandbox modes. The config you choose determines how isolated the agent's workspace is from your host machine.

```
┌──────────────────────────┬───────────────────────────┬────────────────────────┐
│ Provider                 │ Shell access              │ Use case               │
├──────────────────────────┼───────────────────────────┼────────────────────────┤
│ LocalSandboxProvider     │ File tools only (no bash) │ Dev / low-risk tasks   │
│ AioSandboxProvider       │ Containerised bash        │ CI-style coding tasks  │
└──────────────────────────┴───────────────────────────┴────────────────────────┘
```

**Key question:** If an agent tries to run `rm -rf /`, which provider is safe?

- `LocalSandboxProvider` — `allow_bash: false` means no shell at all. The command never executes.
- `AioSandboxProvider` — the command runs inside an isolated container. Your host is safe; the container takes the hit.

> ⚠️ **Security note:** Neither provider makes it safe to expose DeerFlow on an untrusted network. Sandbox ≠ authentication or network isolation.

In [ ]:
# Part A · Cell 3 — Simulate the sandbox provider decision

def choose_provider(allow_bash: bool, trusted_network: bool) -> str:
    if not trusted_network:
        return 'REJECT — never expose sandbox on an untrusted network'
    if allow_bash:
        return 'AioSandboxProvider — bash in an isolated container'
    return 'LocalSandboxProvider — file tools only, no shell'

scenarios = [
    ('Local dev, no bash needed',    False, True),
    ('CI pipeline, run tests',       True,  True),
    ('Exposed to the internet',      True,  False),
]

for label, allow_bash, trusted in scenarios:
    print(f'{label:<35} → {choose_provider(allow_bash, trusted)}')

In [ ]:
# Part A · Cell 4 — SWEAgent client (pure Python, no server call yet)
# This is the client from src/workflow.py — inspect it before running it.

import json
from dataclasses import dataclass, field
from typing import Iterator
import httpx

@dataclass
class SWEAgent:
    base_url: str
    thread_id: str
    _http: httpx.Client = field(
        default_factory=lambda: httpx.Client(timeout=httpx.Timeout(60.0, read=300.0))
    )

    def upload(self, filename: str, content: str) -> str:
        resp = self._http.post(
            f'{self.base_url}/api/files/upload',
            files={'file': (filename, content.encode(), 'text/plain')},
            data={'thread_id': self.thread_id},
        )
        resp.raise_for_status()
        return resp.json().get('artifact_id', 'unknown')

    def stream(self, prompt: str) -> Iterator[tuple[str, dict]]:
        with self._http.stream(
            'POST', f'{self.base_url}/api/chat/stream',
            json={'message': prompt, 'thread_id': self.thread_id, 'plan_mode': False},
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line.startswith('data:'):
                    continue
                raw = line.removeprefix('data:').strip()
                if raw == '[DONE]':
                    yield 'end', {}
                    return
                try:
                    ev = json.loads(raw)
                    yield ev.get('type', 'unknown'), ev
                except json.JSONDecodeError:
                    pass

print('SWEAgent defined — ready for Part B')

---
## Part B · Live DeerFlow Run

The cells below require a running DeerFlow instance.

```bash
# In the deer-flow repo:
cp examples/136-deerflow-sandboxed-swe-agent/runtime/config.aio.yaml conf/config.yaml
make dev
```

Cell 5 will raise immediately if DeerFlow is not reachable — **no silent hangs**.

In [ ]:
# Part B · Cell 5 — Connect (fail fast if DeerFlow is down)
import os
from dotenv import load_dotenv
load_dotenv()

BASE_URL = os.getenv('DEERFLOW_BASE_URL', 'http://localhost:8000')
THREAD_ID = 'notebook-thread-136'

try:
    r = httpx.get(f'{BASE_URL}/api/health', timeout=5)
    r.raise_for_status()
    print(f'✓ DeerFlow reachable at {BASE_URL}')
except Exception as exc:
    raise RuntimeError(
        f'DeerFlow is not reachable at {BASE_URL}.\n'
        'Start it with `make dev` in the deer-flow repo, then re-run this cell.'
    ) from exc

agent = SWEAgent(base_url=BASE_URL, thread_id=THREAD_ID)

In [ ]:
# Part B · Cell 6 — Upload buggy files and run the fix task

BUGGY_SOURCE = 'def add(a, b):\n    return a - b  # BUG\n\ndef multiply(a, b):\n    return a * b\n'
TEST_SOURCE  = 'from calculator import add, multiply\n\ndef test_add():\n    assert add(2, 3) == 5\n\ndef test_multiply():\n    assert multiply(3, 4) == 12\n'
TASK_PROMPT  = (
    'I have uploaded calculator.py and test_calculator.py. '
    'test_add() currently fails because add() subtracts instead of adding. '
    'Fix calculator.py so all tests pass. Return the corrected calculator.py content only.'
)

print('Uploading fixtures…')
for name, src in [('calculator.py', BUGGY_SOURCE), ('test_calculator.py', TEST_SOURCE)]:
    print(f'  {name} → {agent.upload(name, src)}')

print('\nStreaming fix task…\n')
events: list[tuple[str, dict]] = []
for et, data in agent.stream(TASK_PROMPT):
    events.append((et, data))
    if et == 'message_chunk':
        print(data.get('content', ''), end='', flush=True)
    elif et == 'end':
        print()

In [ ]:
# Part B · Cell 7 — Verify the fix

EXPECTED = 'return a + b'
result = ''.join(data.get('content', '') for et, data in events if et == 'message_chunk')
fixed = EXPECTED in result

print(f'Expected fix: {EXPECTED!r}')
print(f'Found in output: {fixed}')
print(f'\n{"✓ PASS" if fixed else "? Check the streamed output above"}')

counts: dict[str, int] = {}
for et, _ in events:
    counts[et] = counts.get(et, 0) + 1
print(f'\nEvent counts: {counts}')
print(f'Thread: {THREAD_ID}  |  Sandbox: AioSandboxProvider (bash in container)')